In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import requests
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

# Indirizzo attuale dell'API Istat
BASE_URL = "https://esploradati.istat.it/SDMXWS/rest"
HEADERS = {"User-Agent": "Mozilla/5.0 (progetto Epicode Data Analyst)"}

# Cartella dove salviamo i file scaricati (il notebook sta in notebook/)
CARTELLA_GREZZI = Path("../dati_grezzi")
CARTELLA_GREZZI.mkdir(exist_ok=True)
print("Cartella dati grezzi:", CARTELLA_GREZZI.resolve())

Cartella dati grezzi: C:\Users\Christian\Desktop\PROGETTO EPICODE\dati_grezzi


In [2]:
FILE_CATALOGO = CARTELLA_GREZZI / "istat_catalogo_dataflow.xml"

if FILE_CATALOGO.exists():
    print("Catalogo già presente: lo leggo dal disco, nessuna richiesta all'Istat.")
    contenuto = FILE_CATALOGO.read_bytes()
else:
    print("Scarico il catalogo dall'Istat...")
    risposta = requests.get(f"{BASE_URL}/dataflow/IT1", headers=HEADERS, timeout=180)
    print("Codice di risposta:", risposta.status_code)  # 200 = tutto ok
    risposta.raise_for_status()  # se c'è un errore, si ferma qui e lo mostra
    contenuto = risposta.content
    FILE_CATALOGO.write_bytes(contenuto)
    print("Catalogo salvato in:", FILE_CATALOGO)

Catalogo già presente: lo leggo dal disco, nessuna richiesta all'Istat.


In [3]:
LINGUA = "{http://www.w3.org/XML/1998/namespace}lang"

righe = []
for elemento in ET.fromstring(contenuto).iter():
    if elemento.tag.endswith("}Dataflow"):
        nomi = {n.attrib.get(LINGUA): n.text for n in elemento if n.tag.endswith("}Name")}
        righe.append({
            "df_id": elemento.attrib.get("id"),
            "descrizione_it": nomi.get("it"),
            "descrizione_en": nomi.get("en"),
        })

catalogo = pd.DataFrame(righe)
print("Dataset disponibili:", len(catalogo))
catalogo.head()

Dataset disponibili: 4955


,df_id,descrizione_it,descrizione_en
0,101_1015,Coltivazioni,Crops
1,101_1015_DF_DCSP_COLTIVAZIONI_1,Superfici e produzione - dati in complesso,Areas and production - overall data
2,101_1015_DF_DCSP_COLTIVAZIONI_10,Intenzioni di semina,Sowing forecast
3,101_1015_DF_DCSP_COLTIVAZIONI_2,Superfici e produzione - dati in complesso - prov,Areas and production - overall data - provinces
4,101_1030,Prodotti agroalimentari di qualità DOP/IGP e STG,"PDO, PGI and TSG quality products"


In [4]:
def cerca(parola):
    """Restituisce i dataset la cui descrizione italiana contiene la parola cercata."""
    trovati = catalogo["descrizione_it"].str.contains(parola, case=False, na=False)
    return catalogo[trovati]

cerca("disoccupazione")

,df_id,descrizione_it,descrizione_en
536,151_1176,Tasso di disoccupazione - dati mensili- regolamento precedente (fino al 2020),Unemployment rate - monthly data- previous regulation (until 2020)
537,151_1176_DF_DCCV_TAXDISOCCUMENS1_UNT2020,Tasso di disoccupazione - dati mensili - regolamento precedente (fino al 2020),Unemployment rate - monthly data - previous regulation (until 2020)
541,151_1178,Tasso di disoccupazione - dati trimestrali destagionalizzati - regolamento precedente (fino al 2020),Unemployment rate - quarterly seasonally adjusted data - previous regulation (until 2020)
550,151_1192_DF_DCCV_DISOCCUPT1_UNT2020_11,Dal IV 1992 - durata della disoccupazione,Since IV 1992 - duration of unemployment
553,151_1192_DF_DCCV_DISOCCUPT1_UNT2020_3,"Condizione professionale, disoccupazione di lunga durata","Labour status, long term unemployment"
560,151_1193,Tasso di disoccupazione - regolamento precedente (fino al 2020),Unemployment rate - previous regulation (until 2020)
565,151_1193_DF_DCCV_TAXDISOCCU1_UNT2020_13,"Dal IV 1992 - dati regionali, durata della disoccupazione","Since IV 1992- regional data, duration of unemployment"
567,151_1193_DF_DCCV_TAXDISOCCU1_UNT2020_3,Durata della disoccupazione,Duration of unemployment
570,151_1193_DF_DCCV_TAXDISOCCU1_UNT2020_6,Dati regionali - durata della disoccupazione,Regional data - duration of unemployment
574,151_874,Tasso di disoccupazione - dati mensili,Unemployment rate - monthly data


In [5]:
cerca("popolazione")

,df_id,descrizione_it,descrizione_en
540,151_1177_DF_DCCV_DISOCCUPTMENS1_UNT20202,Incidenza dei disoccupati sulla popolazione,Unemployment ratio - monthly data
578,151_877_DF_DCCV_DISOCCUPTMENS1_2,Incidenza dei disoccupati sulla popolazione,Unemployment ratio - monthly data
789,163_88_DF_DCCN_OCCQSEC2010_2,Popolazione media del periodo,Quarterly average population
790,164_164,Popolazione residente ricostruita - Anni 2002-2019,Estimated resident population - Years 2002-2019
816,164_279,Popolazione residente ricostruita - Anni 1982-1991,Estimated resident population - Years 1982-1991
...,...,...,...
4921,DF_DCSS_RESPOP_PL1Y_4_GC,Popolazione residente per cittadinanza e luogo di dimora abituale 1 anno prima - province e grandi comuni,Resident population by citizenship and place of usual residence one year prior - provinces and large cities
4922,DF_DCSS_RESPOP_PL1Y_5_GC,Popolazione residente per grado di istruzione e luogo di dimora abituale 1 anno prima rispetto alla dimora attuale - province e grandi comuni,Resident population by educational attainment and place of usual residence one year prior compared to the current place of residence - provinces and large cities
4923,DF_DCSS_RESPOP_PL1Y_6_GC,Popolazione residente per condizione professionale e luogo di dimora abituale 1 anno prima rispetto alla dimora attuale - province e grandi comuni,Resident population by current activity status and place of usual residence one year prior compared to the current place of residence - provinces and large cities
4946,POP,Popolazione,Population


In [6]:
def scarica_struttura(df_id):
    """Scarica (una volta sola) la struttura di un dataflow e la salva su disco."""
    file = CARTELLA_GREZZI / f"istat_struttura_{df_id}.xml"
    if file.exists():
        print("Struttura già presente:", file.name)
        return file.read_bytes()
    print("Scarico la struttura di", df_id, "...")
    risposta = requests.get(f"{BASE_URL}/dataflow/IT1/{df_id}",
                            params={"references": "all"}, headers=HEADERS, timeout=300)
    print("Codice di risposta:", risposta.status_code)
    risposta.raise_for_status()
    file.write_bytes(risposta.content)
    return risposta.content


def leggi_struttura(contenuto):
    radice = ET.fromstring(contenuto)
    dimensioni = []
    for el in radice.iter():
        if el.tag.endswith("}Dimension") or el.tag.endswith("}TimeDimension"):
            codelist = None
            for sotto in el.iter():
                if sotto.tag.endswith("Enumeration"):
                    for ref in sotto.iter():
                        if ref.tag.endswith("Ref"):
                            codelist = ref.attrib.get("id")
            dimensioni.append({
                "posizione": int(el.attrib.get("position", 0)),
                "dimensione": el.attrib.get("id"),
                "codelist": codelist,
            })
    codici = []
    for cl in radice.iter():
        if cl.tag.endswith("}Codelist"):
            for codice in cl:
                if codice.tag.endswith("}Code"):
                    nomi = {n.attrib.get(LINGUA): n.text for n in codice if n.tag.endswith("}Name")}
                    codici.append({
                        "codelist": cl.attrib.get("id"),
                        "codice": codice.attrib.get("id"),
                        "nome_it": nomi.get("it"),
                    })
    tab_dim = pd.DataFrame(dimensioni).drop_duplicates().sort_values("posizione").reset_index(drop=True)
    return tab_dim, pd.DataFrame(codici)

def codici_province(codici, dimensioni):
    lista_territorio = dimensioni.loc[dimensioni["codelist"].fillna("").str.contains("ITTER"), "codelist"]
    territorio = codici[codici["codelist"].isin(lista_territorio)]
    trovati = territorio[territorio["nome_it"].isin(PROVINCE)]
    return trovati.sort_values(["nome_it", "codice"])

PROVINCE = ["Milano", "Bergamo", "Pavia", "Roma", "Latina", "Frosinone",
            "Napoli", "Salerno", "Caserta", "Bari", "Lecce", "Foggia"]


def mostra_codici(codici, dimensioni, max_codici=15):
    """Per ogni dimensione (tranne territorio e tempo) mostra i primi codici disponibili."""
    for _, riga in dimensioni.iterrows():
        cl = riga["codelist"]
        if cl is None or pd.isna(cl) or "ITTER" in cl:
            continue
        valori = codici[codici["codelist"] == cl]
        print(f"\n▶ {riga['dimensione']} ({cl}) – {len(valori)} codici")
        print(valori[["codice", "nome_it"]].head(max_codici).to_string(index=False))

print("Funzioni pronte ✅")


Funzioni pronte ✅


In [7]:
ID_DISOCC = "151_914_DF_DCCV_TAXDISOCCU1_8"
dim_disocc, codici_disocc = leggi_struttura(scarica_struttura(ID_DISOCC))
dim_disocc

Struttura già presente: istat_struttura_151_914_DF_DCCV_TAXDISOCCU1_8.xml


,posizione,dimensione,codelist
0,0,NaN,NaN
1,1,FREQ,CL_FREQ
2,2,REF_AREA,CL_ITTER107
3,3,DATA_TYPE,CL_TIPO_DATO_FOL
4,4,SEX,CL_SEXISTAT1
5,5,AGE,CL_ETA1
6,6,EDU_LEV_HIGHEST,CL_TITOLO_STUDIO
7,7,CITIZENSHIP,CL_CITTADINANZA
8,8,DURATION_UNEMPLOYMENT,CL_DURATA
9,9,TIME_PERIOD,NaN


In [8]:
codici_province(codici_disocc, dim_disocc)

,codelist,codice,nome_it
2185,CL_ITTER107,072006,Bari
733,CL_ITTER107,ITF42,Bari
11804,CL_ITTER107,SLL_2011_1612,Bari
12488,CL_ITTER107,SLL_2021_1613,Bari
11163,CL_ITTER107,SLL_457,Bari
2318,CL_ITTER107,016024,Bergamo
626,CL_ITTER107,ITC46,Bergamo
11449,CL_ITTER107,SLL_2011_315,Bergamo
12193,CL_ITTER107,SLL_2021_313,Bergamo
10765,CL_ITTER107,SLL_60,Bergamo


In [9]:
mostra_codici(codici_disocc, dim_disocc)


▶ FREQ (CL_FREQ) – 8 codici
codice                   nome_it
     A                   annuale
     B business (non supportato)
     D               giornaliero
     E    event (non supportato)
     H                semestrale
     M                   mensile
     Q               trimestrale
     W               settimanale

▶ DATA_TYPE (CL_TIPO_DATO_FOL) – 89 codici
codice                                                                    nome_it
   EMP                                        occupati 15 anni e oltre (migliaia)
 UNEMP                 persone in cerca di occupazione 15 anni e oltre (migliaia)
   FOR                                 forze di lavoro 15 anni e oltre (migliaia)
  INAC                                                        inattivi (migliaia)
 EMP_R                                                       tasso di occupazione
UNEM_R                                                    tasso di disoccupazione
 ACT_R                                                  

In [10]:
ID_POP = "22_289"
dim_pop, codici_pop = leggi_struttura(scarica_struttura(ID_POP))
dim_pop

Struttura già presente: istat_struttura_22_289.xml


,posizione,dimensione,codelist
0,0,NaN,NaN
1,1,FREQ,CL_FREQ
2,2,REF_AREA,CL_ITTER107
3,3,DATA_TYPE,CL_TIPO_DATO15
4,4,SEX,CL_SEXISTAT1
5,5,AGE,CL_ETA1
6,6,MARITAL_STATUS,CL_STATCIV2
7,7,TIME_PERIOD,NaN


In [11]:
codici_province(codici_pop, dim_pop)

,codelist,codice,nome_it
2027,CL_ITTER107,072006,Bari
575,CL_ITTER107,ITF42,Bari
11646,CL_ITTER107,SLL_2011_1612,Bari
12330,CL_ITTER107,SLL_2021_1613,Bari
11005,CL_ITTER107,SLL_457,Bari
2160,CL_ITTER107,016024,Bergamo
468,CL_ITTER107,ITC46,Bergamo
11291,CL_ITTER107,SLL_2011_315,Bergamo
12035,CL_ITTER107,SLL_2021_313,Bergamo
10607,CL_ITTER107,SLL_60,Bergamo


In [12]:
mostra_codici(codici_pop, dim_pop)


▶ FREQ (CL_FREQ) – 8 codici
codice                   nome_it
     A                   annuale
     B business (non supportato)
     D               giornaliero
     E    event (non supportato)
     H                semestrale
     M                   mensile
     Q               trimestrale
     W               settimanale

▶ DATA_TYPE (CL_TIPO_DATO15) – 476 codici
   codice                                                  nome_it
  POPSTAT                     popolazione statistica al 1º gennaio
      JAN                                popolazione al 1º gennaio
    JAN_C          popolazione al 1º gennaio ai confini dell'epoca
      BEG                        popolazione censita al 1° gennaio
   LBIRTH                                                nati vivi
 LBIRTHES                                      stima dei nati vivi
LBIRTHESP                 stima dei nati vivi (valori percentuali)
    DEATH                                                    morti
   NTGROW                   

In [13]:
# Codici Istat delle 12 province: codice territorio API -> codice provincia a 3 cifre
# Il codice a 3 cifre è la chiave che useremo per unire Istat, MEF e OMI.
MAPPA_PROVINCE = {
    "ITC45": ("Milano", "015"),    "ITC46": ("Bergamo", "016"),  "ITC48": ("Pavia", "018"),
    "ITE43": ("Roma", "058"),      "ITE44": ("Latina", "059"),   "ITE45": ("Frosinone", "060"),
    "ITF33": ("Napoli", "063"),    "ITF35": ("Salerno", "065"),  "ITF31": ("Caserta", "061"),
    "ITF42": ("Bari", "072"),      "ITF45": ("Lecce", "075"),    "ITF41": ("Foggia", "071"),
}
AREE = "+".join(MAPPA_PROVINCE.keys())
ACCEPT_CSV = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}


def scarica_dati(df_id, chiave, nome_file, anno_inizio):
    """Scarica (una volta sola) i dati di un dataflow in CSV e li salva su disco."""
    file = CARTELLA_GREZZI / nome_file
    if file.exists():
        print("Dati già presenti:", file.name)
    else:
        url = f"{BASE_URL}/data/{df_id}/{chiave}"
        print("Richiesta:", url)
        risposta = requests.get(url, params={"startPeriod": anno_inizio},
                                headers={**HEADERS, **ACCEPT_CSV}, timeout=300)
        print("Codice di risposta:", risposta.status_code)
        if risposta.status_code != 200:
            print(risposta.text[:500])  # mostra il messaggio d'errore dell'Istat
        risposta.raise_for_status()
        file.write_bytes(risposta.content)
        print("Salvato in:", file)
    return pd.read_csv(file, dtype=str)  # tutto come testo: i codici non perdono gli zeri


def esplora_colonne(df):
    """Per ogni colonna mostra quanti valori diversi ha e quali sono (se pochi)."""
    for col in df.columns:
        valori = df[col].dropna().unique()
        anteprima = list(valori[:12]) if len(valori) <= 12 else list(valori[:6]) + ["..."]
        print(f"{col:<28} {len(valori):>5} valori  {anteprima}")

print("Province:", len(MAPPA_PROVINCE))
print("Chiave territorio:", AREE)

Province: 12
Chiave territorio: ITC45+ITC46+ITC48+ITE43+ITE44+ITE45+ITF33+ITF35+ITF31+ITF42+ITF45+ITF41


In [14]:
chiave_disocc = ".".join(["A", AREE, "UNEM_R", "", "", "", "", ""])
df_disocc = scarica_dati(ID_DISOCC, chiave_disocc, "istat_disoccupazione.csv", 2018)
print("Righe scaricate:", len(df_disocc))
df_disocc.head()

Richiesta: https://esploradati.istat.it/SDMXWS/rest/data/151_914_DF_DCCV_TAXDISOCCU1_8/A.ITC45+ITC46+ITC48+ITE43+ITE44+ITE45+ITF33+ITF35+ITF31+ITF42+ITF45+ITF41.UNEM_R.....
Codice di risposta: 200
Salvato in: ..\dati_grezzi\istat_disoccupazione.csv
Righe scaricate: 2301


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,SEX,AGE,EDU_LEV_HIGHEST,CITIZENSHIP,DURATION_UNEMPLOYMENT,TIME_PERIOD,...,NOTE_DATA_TYPE,NOTE_SEX,NOTE_AGE,NOTE_EDU_LEV_HIGHEST,NOTE_CITIZENSHIP,NOTE_DURATION_UNEMPLOYMENT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0),A,ITC45,UNEM_R,1,Y15-24,99,TOTAL,TOTAL,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0),A,ITC45,UNEM_R,1,Y15-24,99,TOTAL,TOTAL,2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0),A,ITC45,UNEM_R,1,Y15-24,99,TOTAL,TOTAL,2020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0),A,ITC45,UNEM_R,1,Y15-24,99,TOTAL,TOTAL,2021,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0),A,ITC45,UNEM_R,1,Y15-24,99,TOTAL,TOTAL,2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
esplora_colonne(df_disocc)

DATAFLOW                         1 valori  ['IT1:151_914_DF_DCCV_TAXDISOCCU1_8(1.0)']
FREQ                             1 valori  ['A']
REF_AREA                        12 valori  ['ITC45', 'ITC46', 'ITC48', 'ITE43', 'ITE44', 'ITE45', 'ITF31', 'ITF33', 'ITF35', 'ITF41', 'ITF42', 'ITF45']
DATA_TYPE                        1 valori  ['UNEM_R']
SEX                              3 valori  ['1', '2', '9']
AGE                              8 valori  ['Y15-24', 'Y15-34', 'Y15-64', 'Y15-74', 'Y20-64', 'Y25-34', 'Y35-49', 'Y50-74']
EDU_LEV_HIGHEST                  1 valori  ['99']
CITIZENSHIP                      1 valori  ['TOTAL']
DURATION_UNEMPLOYMENT            1 valori  ['TOTAL']
TIME_PERIOD                      8 valori  ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
OBS_VALUE                     2296 valori  ['25.412512', '17.265643', '19.739375', '23.169057', '15.819508', '20.120456', '...']
OBS_STATUS                       0 valori  []
NOTE_DS                          0 va

In [18]:
def codice_totale(codici, codelist):
    """Cerca nella codelist il codice che corrisponde a 'totale'."""
    cl = codici[codici["codelist"] == codelist]
    trovati = cl[cl["nome_it"].fillna("").str.strip().str.lower() == "totale"]
    print(f"{codelist:<14} -> {list(trovati['codice'])}")
    return trovati["codice"].iloc[0] if len(trovati) else ""

sesso_tot = codice_totale(codici_pop, "CL_SEXISTAT1")
eta_tot = codice_totale(codici_pop, "CL_ETA1")
statociv_tot = codice_totale(codici_pop, "CL_STATCIV2")

chiave_pop = ".".join(["A", AREE, "JAN", sesso_tot, eta_tot, statociv_tot])
print("Chiave:", chiave_pop)
df_pop = scarica_dati(ID_POP, chiave_pop, "istat_popolazione.csv", 2018)
print("Righe scaricate:", len(df_pop))
df_pop.head()

CL_SEXISTAT1   -> ['9', 'T']
CL_ETA1        -> ['TOTAL']
CL_STATCIV2    -> ['99', 'ALL']
Chiave: A.ITC45+ITC46+ITC48+ITE43+ITE44+ITE45+ITF33+ITF35+ITF31+ITF42+ITF45+ITF41.JAN.9.TOTAL.99
Richiesta: https://esploradati.istat.it/SDMXWS/rest/data/22_289/A.ITC45+ITC46+ITC48+ITE43+ITE44+ITE45+ITF33+ITF35+ITF31+ITF42+ITF45+ITF41.JAN.9.TOTAL.99
Codice di risposta: 200
Salvato in: ..\dati_grezzi\istat_popolazione.csv
Righe scaricate: 96


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,SEX,AGE,MARITAL_STATUS,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_SEX,NOTE_AGE,NOTE_MARITAL_STATUS,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:22_289(1.0),A,ITC45,JAN,9,TOTAL,99,2019,3250077,NaN,NaN,FILTER__ITC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:22_289(1.0),A,ITC45,JAN,9,TOTAL,99,2020,3265327,NaN,NaN,FILTER__ITC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:22_289(1.0),A,ITC45,JAN,9,TOTAL,99,2021,3241813,NaN,NaN,FILTER__ITC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:22_289(1.0),A,ITC45,JAN,9,TOTAL,99,2022,3214630,NaN,NaN,FILTER__ITC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:22_289(1.0),A,ITC45,JAN,9,TOTAL,99,2023,3228006,NaN,NaN,FILTER__ITC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
esplora_colonne(df_pop)

DATAFLOW                         1 valori  ['IT1:22_289(1.0)']
FREQ                             1 valori  ['A']
REF_AREA                        12 valori  ['ITC45', 'ITC46', 'ITC48', 'ITE43', 'ITE44', 'ITE45', 'ITF31', 'ITF33', 'ITF35', 'ITF41', 'ITF42', 'ITF45']
DATA_TYPE                        1 valori  ['JAN']
SEX                              1 valori  ['9']
AGE                              1 valori  ['TOTAL']
MARITAL_STATUS                   1 valori  ['99']
TIME_PERIOD                      8 valori  ['2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
OBS_VALUE                       96 valori  ['3250077', '3265327', '3241813', '3214630', '3228006', '3245459', '...']
OBS_STATUS                       1 valori  ['e']
NOTE_DS                          0 valori  []
NOTE_REF_AREA                    4 valori  ['FILTER__ITC4', 'FILTER__ITE4', 'FILTER__ITF3', 'FILTER__ITF4']
NOTE_DATA_TYPE                   0 valori  []
NOTE_SEX                         0 valori  []
NOTE_AGE     

In [20]:
# --- 1. Disoccupazione: teniamo solo sesso = totale ed età = 15-74 anni ---
disocc = df_disocc[(df_disocc["SEX"] == "9") & (df_disocc["AGE"] == "Y15-74")]
disocc = disocc[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
    "REF_AREA": "codice_territorio", "TIME_PERIOD": "anno", "OBS_VALUE": "tasso_disoccupazione"})

# --- 2. Popolazione: è già filtrata sui totali, teniamo le colonne utili ---
pop = df_pop[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
    "REF_AREA": "codice_territorio", "TIME_PERIOD": "anno", "OBS_VALUE": "popolazione"})

# --- 3. Da testo a numero ---
disocc["tasso_disoccupazione"] = disocc["tasso_disoccupazione"].astype(float).round(1)
pop["popolazione"] = pop["popolazione"].astype(int)
disocc["anno"] = disocc["anno"].astype(int)
pop["anno"] = pop["anno"].astype(int)

# --- 4. Uniamo le due tabelle per territorio e anno ---
# how="outer": teniamo anche gli anni presenti in una sola delle due tabelle
istat = pd.merge(disocc, pop, on=["codice_territorio", "anno"], how="outer")
istat["popolazione"] = istat["popolazione"].astype("Int64")  # intero che ammette valori mancanti

# --- 5. Aggiungiamo nome, codice a 3 cifre e regione ---
REGIONI = {"ITC4": "Lombardia", "ITE4": "Lazio", "ITF3": "Campania", "ITF4": "Puglia"}
istat["provincia"] = istat["codice_territorio"].map(lambda c: MAPPA_PROVINCE[c][0])
istat["codice_provincia"] = istat["codice_territorio"].map(lambda c: MAPPA_PROVINCE[c][1])
istat["regione"] = istat["codice_territorio"].str[:4].map(REGIONI)

istat = istat[["codice_provincia", "provincia", "regione", "anno",
               "tasso_disoccupazione", "popolazione"]].sort_values(["codice_provincia", "anno"])

# --- 6. Controlli ---
print("Righe:", len(istat), "| Province:", istat["provincia"].nunique())
print("Valori mancanti per colonna:")
print(istat.isna().sum())

# --- 7. Salviamo nella cartella dei dati puliti ---
CARTELLA_PULITI = Path("../dati_puliti")
CARTELLA_PULITI.mkdir(exist_ok=True)
istat.to_csv(CARTELLA_PULITI / "istat_province.csv", index=False)
print("Salvato in:", CARTELLA_PULITI / "istat_province.csv")

# --- 8. Anteprima: l'ultimo anno con entrambi i dati ---
ultimo_anno = istat.dropna()["anno"].max()
print(f"\nSituazione nel {ultimo_anno}:")
istat[istat["anno"] == ultimo_anno].sort_values("tasso_disoccupazione")

Righe: 108 | Province: 12
Valori mancanti per colonna:
codice_provincia         0
provincia                0
regione                  0
anno                     0
tasso_disoccupazione    12
popolazione             12
dtype: int64
Salvato in: ..\dati_puliti\istat_province.csv

Situazione nel 2025:


,codice_provincia,provincia,regione,anno,tasso_disoccupazione,popolazione
16,016,Bergamo,Lombardia,2025,1.3,1114780
7,015,Milano,Lombardia,2025,3.0,3246455
25,018,Pavia,Lombardia,2025,4.1,541949
34,058,Roma,Lazio,2025,5.3,4222648
52,060,Frosinone,Lazio,2025,5.3,462661
43,059,Latina,Lazio,2025,5.9,566698
97,072,Bari,Puglia,2025,6.3,1219545
79,065,Salerno,Campania,2025,9.8,1055372
106,075,Lecce,Puglia,2025,10.2,764491
61,061,Caserta,Campania,2025,11.3,908451
